In [ ]:
import os
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

os.environ['GROQ_API_KEY'] = 'your-key-here'

# ── Load data ─────────────────────────────────────────────────
df = pd.read_csv('/content/sample_data/kaggle_london_house_price_data.csv')
df = df[df['saleEstimate_currentPrice'].notna()].head(50)

# ── Build individual property documents ──────────────────────
raw_documents = []
for _, row in df.iterrows():
    price = row['saleEstimate_currentPrice']
    area  = str(row.get('outcode', 'Unknown'))
    sqm   = row.get('floorAreaSqM', 0)
    ptype = row.get('propertyType', 'Unknown')

    ptype = str(ptype) if pd.notna(ptype) else 'Unknown'
    area  = str(area)  if pd.notna(area)  else 'Unknown'
    sqm   = float(sqm) if pd.notna(sqm) and sqm != 0 else None
    ppsqm = round(price / sqm, 0) if sqm else 'Unknown'

    content = (
        f"Property type: {ptype}. "
        f"Location: {area}. "
        f"Price: £{price:,.0f}. "
        f"Floor area: {sqm} sqm. "
        f"Price per sqm: £{ppsqm}. "
        f"This is a {ptype.lower()} property in {area} "
        f"listed at £{price:,.0f}."
    )
    raw_documents.append(Document(
        page_content=content,
        metadata={
            "area":   area,
            "price":  price,
            "type":   ptype,
            "source": "london_property_data.csv"
        }
    ))

# ── Add pre-computed summary document ────────────────────────
area_counts   = df['outcode'].value_counts().head(10)
area_summary  = ", ".join([
    f"{a}: {c} properties" for a, c in area_counts.items()
])
summary_doc = Document(
    page_content=(
        f"Property count by area in dataset: {area_summary}. "
        f"Total properties: {len(df)}."
    ),
    metadata={"area":"summary","price":0,
              "type":"summary","source":"london_property_data.csv"}
)
raw_documents.append(summary_doc)

# ── Add pre-computed statistics document ─────────────────────
valid_sqm = df[
    df['floorAreaSqM'].notna() & (df['floorAreaSqM'] > 0)
].copy()
valid_sqm['ppsqm'] = (
    valid_sqm['saleEstimate_currentPrice'] /
    valid_sqm['floorAreaSqM']
)
avg_ppsqm = round(valid_sqm['ppsqm'].mean(), 0)
avg_price = round(df['saleEstimate_currentPrice'].mean(), 0)
min_price = round(df['saleEstimate_currentPrice'].min(), 0)
max_price = round(df['saleEstimate_currentPrice'].max(), 0)

# ── Add property type counts to stats doc ────────────────────
type_counts  = df['propertyType'].fillna('Unknown').value_counts()
type_total   = type_counts.sum()
type_summary = ", ".join([
    f"{t}: {c}" for t, c in type_counts.items()
])

# Run this first to see what your data actually has
print("Property type breakdown:")
print(df['propertyType'].fillna('Unknown').value_counts())
print(f"Total: {df['propertyType'].fillna('Unknown').value_counts().sum()}")

stats_doc = Document(
    page_content=(
        f"Overall statistics for all 50 London properties: "
        f"Average price £{avg_price:,.0f}. "
        f"Minimum price £{min_price:,.0f}. "
        f"Maximum price £{max_price:,.0f}. "
        f"Average price per sqm £{avg_ppsqm:,.0f}. "
        f"Total properties analysed: {len(df)}. "
        f"Property type breakdown covering all {type_total} properties: "
        f"{type_summary}. "
        f"The most common property type is "
        f"{type_counts.index[0]} with {type_counts.iloc[0]} properties."
    ),
    metadata={"area":"statistics","price":0,
              "type":"statistics","source":"london_property_data.csv"}
)
raw_documents.append(stats_doc)

print(f"✓ {len(df)} properties + 2 summary docs "
      f"= {len(raw_documents)} total documents")

# ── Chunk and build vector store ─────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
documents   = splitter.split_documents(raw_documents)
embeddings  = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever   = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10}
)
print("✓ Vector store built with MMR retrieval")

# ── RAG chain ─────────────────────────────────────────────────
model  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
parser = StrOutputParser()

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a London property investment advisor.
Answer using ONLY the property data provided below.
Be specific — mention exact prices, areas, and property types.
For questions about counts or averages use the statistics
and summary documents provided.
After your answer add: Sources: [areas or documents referenced]
If data is insufficient say: I don't have that information.
Never invent property details.

Property data:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n".join([
        f"- [{doc.metadata.get('area','?')}] {doc.page_content}"
        for doc in docs
    ])

rag_chain = (
    {"context":  retriever | format_docs,
     "question": RunnablePassthrough()}
    | rag_prompt | model | parser
)

# ── Run all 5 questions ───────────────────────────────────────
questions = [
    "Which areas have the most properties available?",
    "What is the average price per sqm across all properties?",
    "Which property type appears most frequently?",
    "What is the most expensive property and where?",
    "Which areas offer the best value based on price per sqm?"
]

print("\n" + "="*55)
print("London Property Q&A Assistant v2 — Production Ready")
print("="*55)

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")

# ── Save ──────────────────────────────────────────────────────
vectorstore.save_local("property_index_v2")
print("\n✓ Upgraded vector store saved to property_index_v2/")